In [25]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [26]:
!pip install neuralforecast

In [27]:
from neuralforecast import NeuralForecast
from neuralforecast.models import PatchTST
import pandas as pd
import numpy as np

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

import matplotlib.pyplot as plt

In [28]:
from pathlib import Path


def load_processed_data(data_path, store_code, channel):

    file_path = Path(data_path) / store_code / f"{channel}.parquet"

    df = pd.read_parquet(file_path)

    return df

In [29]:
def prepare_nhits_data(df):


    patch_df = pd.DataFrame()

    # -------------------------
    # Target
    # -------------------------

    patch_df["ds"] = pd.to_datetime(df["transaction_date"])
    patch_df["y"] = df["Total_Sales"]

    # -------------------------
    # Time Features
    # -------------------------

    patch_df["Is_Weekend"] = df["Is_Weekend"]

    patch_df["Day_of_Week"] = (
        patch_df["ds"].dt.dayofweek + 1
    )

    patch_df["Day_of_Month"] = (
        patch_df["ds"].dt.day
    )

    patch_df["unique_id"] = "Sales"

    patch_df["ds"] = pd.to_datetime(patch_df["ds"])


    '''
    arima_df["Week_of_Year"] = (
        arima_df["ds"].dt.isocalendar().week.astype(int)
    )

    arima_df["Month"] = (
        arima_df["ds"].dt.month
    )'''

    return patch_df




In [30]:
def prepare_prophet_data(df):

    prophet_df = pd.DataFrame()

    prophet_df["ds"] = pd.to_datetime(
        df["transaction_date"]
    )

    prophet_df["y"] = df["Total_Sales"]

    prophet_df["Is_Weekend"] = df["Is_Weekend"]
    return prophet_df


In [31]:
EVENTS = {

    # ======================================================
    # RAMADAN CAMPAIGNS
    # ======================================================

    "Ramadan_1": [
        ("2023-02-16","2023-03-02"),
        ("2024-01-25","2024-02-08"),
        ("2025-02-05","2025-02-19"),
        ("2026-01-15","2026-01-29"),
    ],

    "Ramadan_2": [
        ("2023-03-03","2023-03-16"),
        ("2024-02-09","2024-02-22"),
        ("2025-02-20","2025-03-05"),
        ("2026-01-30","2026-02-12"),
    ],

    "Ramadan_3": [
        ("2023-03-17","2023-03-30"),
        ("2024-02-23","2024-03-07"),
        ("2025-03-06","2025-03-19"),
        ("2026-02-13","2026-02-26"),
    ],

    # ======================================================
    # EID EL FITR PROMOTION
    # ======================================================

    "Eid_Fitr": [
        ("2023-03-31","2023-04-14"),
        ("2024-03-08","2024-03-22"),
        ("2025-03-20","2025-04-03"),
        ("2026-02-27","2026-03-13"),
    ],

    # ======================================================
    # EID EL ADHA PROMOTION
    # ======================================================

    "Eid_Adha": [
        ("2023-06-10","2023-06-28"),
        ("2024-05-19","2024-06-06"),
        ("2025-05-29","2025-06-16"),
        ("2026-05-18","2026-06-05"),
    ],

    # ======================================================
    # SHAM EL NESSIM
    # (15-day promotion ending on Sham El Nessim)
    # ======================================================

    "Sham_El_Nessim": [
        ("2023-04-03","2023-04-17"),
        ("2024-04-22","2024-05-06"),
        ("2025-04-14","2025-04-28"),
        ("2026-03-30","2026-04-13"),
    ],

    # ======================================================
    # BACK TO SCHOOL
    # ======================================================

    "Back_to_School_1": [
        ("2023-08-25","2023-09-10"),
        ("2024-08-25","2024-09-10"),
        ("2025-08-25","2025-09-10"),
        ("2026-08-25","2026-09-10"),
    ],

    "Back_to_School_2": [
        ("2023-09-11","2023-09-28"),
        ("2024-09-11","2024-09-28"),
        ("2025-09-11","2025-09-28"),
        ("2026-09-11","2026-09-28"),
    ],

    # ======================================================
    # GREEN NOVEMBER
    # ======================================================

    "Green_Nov_1": [
        ("2023-11-02","2023-11-12"),
        ("2024-11-02","2024-11-12"),
        ("2025-11-02","2025-11-12"),
        ("2026-11-02","2026-11-12"),
    ],

    "Green_Nov_2": [
        ("2023-11-13","2023-11-24"),
        ("2024-11-13","2024-11-24"),
        ("2025-11-13","2025-11-24"),
        ("2026-11-13","2026-11-24"),
    ],

    "Green_Nov_3": [
        ("2023-11-25","2023-12-04"),
        ("2024-11-25","2024-12-04"),
        ("2025-11-25","2025-12-04"),
        ("2026-11-25","2026-12-04"),
    ],

    # ======================================================
    # CHRISTMAS
    # ======================================================

    "Christmas": [
        ("2022-12-24","2023-01-06"),
        ("2023-12-24","2024-01-06"),
        ("2024-12-24","2025-01-06"),
        ("2025-12-24","2026-01-06"),
        ("2026-12-24","2027-01-06"),
    ],

    # ======================================================
    # MOTHER'S DAY
    # ======================================================

    "Mother_Day": [
        ("2023-03-12","2023-03-24"),
        ("2024-03-12","2024-03-24"),
        ("2025-03-12","2025-03-24"),
        ("2026-03-12","2026-03-24"),
    ],
}

In [32]:
def add_event_features(df):
    """
    Add Active and Progress features for all promotional events.
    """

    data = df.copy()

    data["transaction_date"] = pd.to_datetime(data["ds"])

    # -----------------------------------------
    # Initialize Columns
    # -----------------------------------------

    for event in EVENTS.keys():

        data[f"{event}_Active"] = 0

        data[f"{event}_Progress"] = 0.0

    # -----------------------------------------
    # Fill Events
    # -----------------------------------------

    for event, periods in EVENTS.items():

        for start, end in periods:

            start = pd.Timestamp(start)
            end = pd.Timestamp(end)

            mask = (
                (data["transaction_date"] >= start) &
                (data["transaction_date"] <= end)
            )

            if not mask.any():
                continue

            total_days = (end - start).days + 1

            progress = (
                (
                    data.loc[mask, "transaction_date"] - start
                ).dt.days + 1
            ) / total_days

            data.loc[mask, f"{event}_Active"] = 1

            data.loc[mask, f"{event}_Progress"] = progress.values
    data = data.drop(columns=["transaction_date"])
    return data

In [33]:
def train_validation_test_split(df):
    """
    Split data into:
    Train      : 2023-01-01 -> 2024-12-31
    Validation : 2025-01-01 -> 2025-06-30
    Test        : 2025-07-01 -> 2025-12-31
    """

    data = df.copy()
    data["ds"] = pd.to_datetime(data["ds"])

    train = data[
        data["ds"] < "2026-01-01"
    ].copy()

    validation = data[
        (data["ds"] >= "2026-01-01") &
        (data["ds"] < "2026-07-01")
    ].copy()

    test = data[
        data["ds"] >= "2026-07-01"
    ].copy()

    return train, validation, test

In [34]:
from prophet import Prophet

def train_prophet(train_df):
    """
    Train Prophet model.
    """

    model = Prophet(

    yearly_seasonality=False,

    weekly_seasonality=True,

    daily_seasonality=False,

    seasonality_mode="additive"

    )


    # Weekend كـ External Regressor
    model.fit(train_df)

    return model




In [35]:
def predict_prophet(model, future_df):
    """
    Predict using Prophet.
    """

    future = future_df[["ds"]].copy()

    forecast = model.predict(future)

    return forecast

In [36]:
from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS


def train_nhits(train_df, horizon):

    model = NHITS(

        h=horizon,

        input_size=365,

        #futr_exog_list=FUTURE_EXOG,

        max_steps=500,

        learning_rate=1e-3,

        batch_size=32,

        windows_batch_size=256,

        scaler_type="robust",

        activation="ReLU",

        dropout_prob_theta=0.1,

        mlp_units=[
            [256, 256],
            [256, 256],
            [256, 256]
        ],

        n_blocks=[1,1,1],

        n_pool_kernel_size=[2,2,1],

        n_freq_downsample=[4,2,1]

    )

    nf = NeuralForecast(

        models=[model],

        freq="D"

    )

    nf.fit(train_df)

    return nf

In [37]:
def predict_nhits(model, valid_df):

    forecast = model.predict(

        futr_df=valid_df

    )

    prediction = forecast.copy()

    prediction.rename(

        columns={

            "NHITS":"yhat"

        },

        inplace=True

    )

    return prediction

In [38]:
def combine_forecasts(

    nhits_forecast,
    prophet_forecast,
    nhits_weight=0.2,
    prophet_weight=0.8

):

    final = nhits_forecast.copy()

    final["yhat"] = (

        nhits_weight * nhits_forecast["yhat"] +

        prophet_weight * prophet_forecast["yhat"]

    )

    return final

In [39]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error
)

import numpy as np


def evaluate_forecast(actual, predicted):

    mae = mean_absolute_error(actual, predicted)

    rmse = np.sqrt(
        mean_squared_error(actual, predicted)
    )

    mape = mean_absolute_percentage_error(
        actual,
        predicted
    ) * 100

    return {
        "MAE": mae,
        "RMSE": rmse,
        "MAPE": mape
    }

In [40]:
import matplotlib.pyplot as plt


def plot_forecast(

    valid_df,

    prophet_forecast,

    nhits_forecast,

    final_forecast

):

    fig, ax = plt.subplots(

        figsize=(18,6)

    )

    ax.plot(

        valid_df["ds"],

        valid_df["y"],

        label="Actual",

        linewidth=3

    )

    ax.plot(

        valid_df["ds"],

        prophet_forecast["yhat"],

        label="Prophet"

    )

    ax.plot(

        valid_df["ds"],

        nhits_forecast["yhat"],

        label="NHITS"

    )

    ax.plot(

        valid_df["ds"],

        final_forecast["yhat"],

        label="Ensemble"

    )

    ax.set_title("Forecast Comparison")

    ax.legend()

    ax.grid(True)

    plt.tight_layout()

    return fig

In [53]:
from openpyxl.drawing.image import Image as XLImage
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import tempfile


def save_results_to_excel(

    output_path,

    store_name,

    channel,

    valid_df,

    prophet_forecast,

    nhits_forecast,

    final_forecast,

    prophet_metrics,

    nhits_metrics,

    ensemble_metrics

):

    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:

        # ============================================================
        # Forecast Sheet
        # ============================================================

        forecast_df = pd.DataFrame({

            "Date": valid_df["ds"].values,

            "Actual": valid_df["y"].values,

            "Prophet Prediction":
                prophet_forecast["yhat"].values,

            "NHITS Prediction":
                nhits_forecast["yhat"].values,

            "Ensemble Prediction":
                final_forecast["yhat"].values,

        })

        # ------------------------------------------------------

        forecast_df["Prophet Error"] = (

            forecast_df["Prophet Prediction"]

            - forecast_df["Actual"]

        )

        forecast_df["NHITS Error"] = (

            forecast_df["NHITS Prediction"]

            - forecast_df["Actual"]

        )

        forecast_df["Ensemble Error"] = (

            forecast_df["Ensemble Prediction"]

            - forecast_df["Actual"]

        )

        # ------------------------------------------------------

        forecast_df["Prophet Absolute Error"] = (

            forecast_df["Prophet Error"].abs()

        )

        forecast_df["NHITS Absolute Error"] = (

            forecast_df["NHITS Error"].abs()

        )

        forecast_df["Ensemble Absolute Error"] = (

            forecast_df["Ensemble Error"].abs()

        )

        # ------------------------------------------------------

        forecast_df["Prophet Percentage Error (%)"] = (

            forecast_df["Prophet Absolute Error"]

            / forecast_df["Actual"]

            * 100

        )

        forecast_df["NHITS Percentage Error (%)"] = (

            forecast_df["NHITS Absolute Error"]

            / forecast_df["Actual"]

            * 100

        )

        forecast_df["Ensemble Percentage Error (%)"] = (

            forecast_df["Ensemble Absolute Error"]

            / forecast_df["Actual"]

            * 100

        )

        # ------------------------------------------------------

        forecast_df["Prophet Forecast Type"] = np.where(

            forecast_df["Prophet Error"] > 0,

            "Over Forecast",

            "Under Forecast"

        )

        forecast_df["NHITS Forecast Type"] = np.where(

            forecast_df["NHITS Error"] > 0,

            "Over Forecast",

            "Under Forecast"

        )

        forecast_df["Ensemble Forecast Type"] = np.where(

            forecast_df["Ensemble Error"] > 0,

            "Over Forecast",

            "Under Forecast"

        )

        forecast_df.to_excel(

            writer,

            sheet_name="Forecast",

            index=False

        )

        # ============================================================
        # Metrics Sheet
        # ============================================================

        metrics_df = pd.DataFrame({

            "Metric":[

                "MAE",

                "RMSE",

                "MAPE"

            ],

            "Prophet":[

                prophet_metrics["MAE"],

                prophet_metrics["RMSE"],

                prophet_metrics["MAPE"]

            ],

            "NHITS":[

                nhits_metrics["MAE"],

                nhits_metrics["RMSE"],

                nhits_metrics["MAPE"]

            ],

            "Ensemble":[

                ensemble_metrics["MAE"],

                ensemble_metrics["RMSE"],

                ensemble_metrics["MAPE"]

            ]

        })

        metrics_df.to_excel(

            writer,

            sheet_name="Metrics",

            index=False

        )

        # ============================================================
        # Plot
        # ============================================================

        plt.figure(figsize=(16,6))

        plt.plot(

            valid_df["ds"],

            valid_df["y"],

            label="Actual",

            linewidth=2

        )

        plt.plot(

            valid_df["ds"],

            prophet_forecast["yhat"],

            label="Prophet"

        )

        plt.plot(

            valid_df["ds"],

            nhits_forecast["yhat"],

            label="NHITS"

        )

        plt.plot(

            valid_df["ds"],

            final_forecast["yhat"],

            label="Ensemble",

            linewidth=3

        )

        plt.title(

            f"{store_name} - {channel}"

        )

        plt.xlabel("Date")

        plt.ylabel("Sales")

        plt.grid(True)

        plt.legend()

        temp = tempfile.NamedTemporaryFile(

            suffix=".png",

            delete=False

        )

        plt.savefig(

            temp.name,

            dpi=250,

            bbox_inches="tight"

        )

        plt.close()

        workbook = writer.book

        worksheet = workbook.create_sheet("Plot")

        img = XLImage(temp.name)

        img.width = 1400

        img.height = 500

        worksheet.add_image(img, "A1")

In [ ]:
from pathlib import Path
import gc

# ==========================================================
# PATHS
# ==========================================================

DATA_PATH = "/content/drive/MyDrive/Spinneys/Forecasting/total_Processed_Data"
OUTPUT_PATH = "/content/drive/MyDrive/Spinneys/Forecasting/FinalResults"

Path(OUTPUT_PATH).mkdir(
    parents=True,
    exist_ok=True
)

# ==========================================================
# STORES
# ==========================================================

stores = sorted([

    folder.name

    for folder in Path(DATA_PATH).iterdir()

    if folder.is_dir()

])

# ==========================================================
# CHANNELS
# ==========================================================

channels = [

    "store",

    "instashop",

    "talabat",

    "spinneys_delivery",

    "combined",

    "loyal_store"

]

# ==========================================================
# TOTAL JOBS
# ==========================================================

all_jobs = [

    (store, channel)

    for store in stores

    for channel in channels

]

total_jobs = len(all_jobs)

# ==========================================================
# MAIN LOOP
# ==========================================================

for job_number, (store_code, channel) in enumerate(all_jobs, start=1):

    print("\n")
    print("=" * 100)
    print(f"[{job_number}/{total_jobs}]")
    print(f"Store   : {store_code}")
    print(f"Channel : {channel}")
    print("=" * 100)

    try:

        # ======================================================
        # LOAD DATA
        # ======================================================

        print("Loading Data ...")

        df = load_processed_data(

            DATA_PATH,

            store_code,

            channel

        )

        # ======================================================
        # PREPARE PROPHET
        # ======================================================

        prophet_df = prepare_prophet_data(df)

        prophet_train, prophet_valid, prophet_test = (

            train_validation_test_split(

                prophet_df

            )

        )

        # ======================================================
        # PREPARE NHITS
        # ======================================================

        nhits_df = prepare_nhits_data(df)

        nhits_df = add_event_features(

            nhits_df

        )

        nhits_train, nhits_valid, nhits_test = (

            train_validation_test_split(

                nhits_df

            )

        )

        # ======================================================
        # TRAIN PROPHET
        # ======================================================

        print("-" * 80)
        print(f"Training Prophet -> {store_code}/{channel}")
        print("-" * 80)

        prophet_model = train_prophet(

            prophet_train

        )

        print("✓ Prophet Training Finished")

        prophet_forecast = predict_prophet(

            prophet_model,

            prophet_valid

        )

        print("✓ Prophet Prediction Finished")

        # ======================================================
        # TRAIN NHITS
        # ======================================================

        print("-" * 80)
        print(f"Training NHITS -> {store_code}/{channel}")
        print("-" * 80)

        nhits_model = train_nhits(

            nhits_train,

            horizon=len(nhits_valid)

        )

        print("✓ NHITS Training Finished")

        nhits_forecast = predict_nhits(

            nhits_model,

            nhits_valid

        )

        print("✓ NHITS Prediction Finished")

        # ======================================================
        # HYBRID
        # ======================================================

        final_forecast = combine_forecasts(

            nhits_forecast,

            prophet_forecast,

            nhits_weight=0.2,

            prophet_weight=0.8

        )

        # ======================================================
        # METRICS
        # ======================================================

        prophet_metrics = evaluate_forecast(

            prophet_valid["y"],

            prophet_forecast["yhat"]

        )

        nhits_metrics = evaluate_forecast(

            nhits_valid["y"],

            nhits_forecast["yhat"]

        )

        ensemble_metrics = evaluate_forecast(

            prophet_valid["y"],

            final_forecast["yhat"]

        )

        # ======================================================
        # SAVE
        # ======================================================

        # ======================================================
        # STORE OUTPUT FOLDER
        # ======================================================

        store_output_folder = Path(OUTPUT_PATH) / store_code

        store_output_folder.mkdir(

            parents=True,

            exist_ok=True

        )

        # ======================================================
        # OUTPUT FILE
        # ======================================================

        output_file = (

            store_output_folder

            /

            f"{channel}.xlsx"

        )

        save_results_to_excel(

            output_path=output_file,

            store_name=store_code,

            channel=channel,

            valid_df=prophet_valid,

            prophet_forecast=prophet_forecast,

            nhits_forecast=nhits_forecast,

            final_forecast=final_forecast,

            prophet_metrics=prophet_metrics,

            nhits_metrics=nhits_metrics,

            ensemble_metrics=ensemble_metrics

        )

        print("✓ Excel Saved")

        # ======================================================
        # MEMORY CLEANUP
        # ======================================================

        del df

        del prophet_df

        del nhits_df

        del prophet_train

        del prophet_valid

        del prophet_test

        del nhits_train

        del nhits_valid

        del nhits_test

        del prophet_model

        del nhits_model

        del prophet_forecast

        del nhits_forecast

        del final_forecast

        gc.collect()

        print(f"✓ Finished {store_code}/{channel}")

    except Exception as e:

        print(f"❌ ERROR in {store_code}/{channel}")

        print(e)

        gc.collect()

        continue

print("\n")
print("=" * 100)
print("ALL STORES FINISHED")
print("=" * 100)



[1/210]
Store   : ALX
Channel : store
Loading Data ...
--------------------------------------------------------------------------------
Training Prophet -> ALX/store
--------------------------------------------------------------------------------


INFO:lightning_fabric.utilities.seed:Seed set to 1


✓ Prophet Training Finished
✓ Prophet Prediction Finished
--------------------------------------------------------------------------------
Training NHITS -> ALX/store
--------------------------------------------------------------------------------


INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.callbacks.model_summary:
  | Name                | Type          | Params | Mode 
--------------------------------------------------------------
0 | loss                | MAE           | 0      | train
1 | hist_cat_embeddings | ModuleList    | 0      | train
2 | futr_cat_embeddings | ModuleList    | 0      | train
3 | stat_cat_embeddings | ModuleList    | 0      | train
4 | padder_train        | ConstantPad1d | 0      | train
5 | scaler              | TemporalNorm  | 0      | train
6 | blocks              | ModuleList    | 1.1 M  | train
--------------------------------------------------------------
1.1 M     Trainable params
0         Non-trainable params
1.1 M     Total params
4.571     Total estimated model params size (MB)
46        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Training: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:
Detected KeyboardInterrupt, attempting graceful shutdown ...
ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/call.py", line 49, in _call_and_handle_interrupt
    return trainer_fn(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/trainer.py", line 598, in _fit_impl
    self._run(model, ckpt_path=ckpt_path)
  File "/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/trainer.py", line 1011, in _run
    results = self._run_stage()
              ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/trainer.py", line 1055, in _run_stage
    self.fit_loop.run()
  File "/usr/local/lib/python3.12/dist-packages/pytorch_lightning/loops/fit_loop.py", line 216, in run
    self.advance()
  File "/usr/local/lib/python3.12/dist-packages/pytorch_lightning/loops/fit_loop.py", line 458, in advance
    self.epoch_loop.run(self._data_fetcher)
  File "/usr/local/lib/python3.12

TypeError: object of type 'NoneType' has no len()

In [ ]:
from pathlib import Path
import pandas as pd

# ==========================================================
# PATH
# ==========================================================

OUTPUT_PATH = Path(
    "/content/drive/MyDrive/Spinneys/Forecasting/FinalResults"
)

# ==========================================================
# CHANNELS
# ==========================================================

channels = [
    "store",
    "instashop",
    "talabat",
    "spinneys_delivery",
    "combined",
    "loyal_store"
]

# ==========================================================
# GET STORES
# ==========================================================

stores = sorted([
    folder.name
    for folder in OUTPUT_PATH.iterdir()
    if folder.is_dir()
])

# ==========================================================
# EXPECTED FILES
# ==========================================================

expected = {
    (store, channel)
    for store in stores
    for channel in channels
}

# ==========================================================
# EXISTING FILES
# ==========================================================

existing = set()

for store in stores:

    store_folder = OUTPUT_PATH / store

    for file in store_folder.glob("*.xlsx"):

        channel = file.stem

        if channel in channels:
            existing.add((store, channel))

# ==========================================================
# MISSING FILES
# ==========================================================

missing = sorted(
    expected - existing
)

# ==========================================================
# EXTRA FILES
# ==========================================================

extra = sorted(
    existing - expected
)

# ==========================================================
# SUMMARY
# ==========================================================

print("\n")
print("=" * 80)
print("FINAL RESULTS CHECK")
print("=" * 80)

print(f"Stores Found       : {len(stores)}")
print(f"Channels           : {len(channels)}")
print(f"Expected Files     : {len(expected)}")
print(f"Existing Files     : {len(existing)}")
print(f"Missing Files      : {len(missing)}")
print(f"Extra Files        : {len(extra)}")

# ==========================================================
# MISSING FILES
# ==========================================================

print("\n")
print("=" * 80)
print("MISSING FILES")
print("=" * 80)

if len(missing) == 0:

    print("✓ No missing files. All expected files exist.")

else:

    for store, channel in missing:

        print(
            f"❌ {store}/{channel}.xlsx"
        )

# ==========================================================
# STORE SUMMARY
# ==========================================================

print("\n")
print("=" * 80)
print("STORE SUMMARY")
print("=" * 80)

summary = []

for store in stores:

    expected_count = len(channels)

    existing_count = sum(
        1
        for channel in channels
        if (store, channel) in existing
    )

    missing_count = (
        expected_count -
        existing_count
    )

    summary.append({

        "Store": store,

        "Expected": expected_count,

        "Existing": existing_count,

        "Missing": missing_count

    })

summary_df = pd.DataFrame(summary)

print(
    summary_df.to_string(index=False)
)

# ==========================================================
# STORES WITH MISSING FILES ONLY
# ==========================================================

print("\n")
print("=" * 80)
print("STORES WITH MISSING FILES")
print("=" * 80)

problem_stores = summary_df[
    summary_df["Missing"] > 0
]

if problem_stores.empty:

    print("✓ Every store has all 6 channels.")

else:

    print(
        problem_stores.to_string(index=False)
    )

# ==========================================================
# CHANNEL SUMMARY
# ==========================================================

print("\n")
print("=" * 80)
print("CHANNEL SUMMARY")
print("=" * 80)

channel_summary = []

for channel in channels:

    expected_count = len(stores)

    existing_count = sum(
        1
        for store in stores
        if (store, channel) in existing
    )

    missing_count = (
        expected_count -
        existing_count
    )

    channel_summary.append({

        "Channel": channel,

        "Expected": expected_count,

        "Existing": existing_count,

        "Missing": missing_count

    })

channel_summary_df = pd.DataFrame(
    channel_summary
)

print(
    channel_summary_df.to_string(index=False)
)

# ==========================================================
# SAVE CHECK RESULT
# ==========================================================

check_output = OUTPUT_PATH / "run_check.xlsx"

with pd.ExcelWriter(
    check_output,
    engine="openpyxl"
) as writer:

    summary_df.to_excel(
        writer,
        sheet_name="Store Summary",
        index=False
    )

    channel_summary_df.to_excel(
        writer,
        sheet_name="Channel Summary",
        index=False
    )

    if missing:

        missing_df = pd.DataFrame(
            missing,
            columns=["Store", "Channel"]
        )

        missing_df["Expected File"] = (
            missing_df["Channel"] + ".xlsx"
        )

    else:

        missing_df = pd.DataFrame(
            columns=[
                "Store",
                "Channel",
                "Expected File"
            ]
        )

    missing_df.to_excel(
        writer,
        sheet_name="Missing Files",
        index=False
    )

print("\n")
print("=" * 80)
print("CHECK FINISHED")
print("=" * 80)

print(
    f"✓ Check file saved to:\n{check_output}"
)



FINAL RESULTS CHECK
Stores Found       : 28
Channels           : 6
Expected Files     : 168
Existing Files     : 116
Missing Files      : 52
Extra Files        : 0


MISSING FILES
❌ ALX/talabat.xlsx
❌ ALX2/talabat.xlsx
❌ ARN/talabat.xlsx
❌ CSC/instashop.xlsx
❌ CSC/talabat.xlsx
❌ DOKI/instashop.xlsx
❌ DOKI/talabat.xlsx
❌ ENAWK/instashop.xlsx
❌ ENAWK/talabat.xlsx
❌ ESTD/instashop.xlsx
❌ ESTD/talabat.xlsx
❌ HRG/instashop.xlsx
❌ HRG/spinneys_delivery.xlsx
❌ HRG/talabat.xlsx
❌ HRG2/instashop.xlsx
❌ HRG2/talabat.xlsx
❌ HRM/talabat.xlsx
❌ MAD/instashop.xlsx
❌ MAD/talabat.xlsx
❌ MAX/instashop.xlsx
❌ MAX/talabat.xlsx
❌ MAZ/instashop.xlsx
❌ MAZ/talabat.xlsx
❌ MHND/instashop.xlsx
❌ MHND/talabat.xlsx
❌ MHND2/instashop.xlsx
❌ MHND2/talabat.xlsx
❌ MIN/instashop.xlsx
❌ MIN/talabat.xlsx
❌ MOA/instashop.xlsx
❌ MOA/talabat.xlsx
❌ MOK/instashop.xlsx
❌ MOK/talabat.xlsx
❌ NCT/instashop.xlsx
❌ NCT/talabat.xlsx
❌ NCT2/spinneys_delivery.xlsx
❌ NCT2/talabat.xlsx
❌ NOZHA/instashop.xlsx
❌ NOZHA/talabat.xlsx
❌ 

In [ ]:
from pathlib import Path
import gc


# ==========================================================
# PATHS
# ==========================================================

DATA_PATH = "/content/drive/MyDrive/Spinneys/Forecasting/total_Processed_Data"
OUTPUT_PATH = "/content/drive/MyDrive/Spinneys/Forecasting/FinalResults"

Path(OUTPUT_PATH).mkdir(
    parents=True,
    exist_ok=True
)


# ==========================================================
# STORES
# ==========================================================

stores = sorted([

    folder.name

    for folder in Path(DATA_PATH).iterdir()

    if folder.is_dir()

])


# ==========================================================
# CHANNELS
# ==========================================================

channels = [

    "store",
    "instashop",
    "talabat",
    "spinneys_delivery",
    "combined",
    "loyal_store"

]


# ==========================================================
# COUNTERS
# ==========================================================

completed_jobs = []
skipped_jobs = []
failed_jobs = []


# ==========================================================
# TOTAL JOBS
# ==========================================================

all_jobs = [

    (store, channel)

    for store in stores

    for channel in channels

]

total_jobs = len(all_jobs)


print("=" * 100)
print(f"TOTAL STORES   : {len(stores)}")
print(f"TOTAL CHANNELS  : {len(channels)}")
print(f"TOTAL JOBS      : {total_jobs}")
print("=" * 100)


# ==========================================================
# MAIN LOOP
# ==========================================================

for job_number, (store_code, channel) in enumerate(
    all_jobs,
    start=1
):

    print("\n")
    print("=" * 100)
    print(f"[{job_number}/{total_jobs}]")
    print(f"Store   : {store_code}")
    print(f"Channel : {channel}")
    print("=" * 100)


    # ======================================================
    # CHECK OUTPUT FILE BEFORE DOING ANYTHING
    # ======================================================

    store_output_folder = (
        Path(OUTPUT_PATH) / store_code
    )

    output_file = (
        store_output_folder / f"{channel}.xlsx"
    )


    # ======================================================
    # SKIP IF ALREADY EXISTS
    # ======================================================

    if output_file.exists():

        print(
            f"⏭️ SKIPPED -> {store_code}/{channel}"
        )

        print(
            f"Already exists: {output_file}"
        )

        skipped_jobs.append(
            (store_code, channel)
        )

        continue


    # ======================================================
    # IF STORE FOLDER DOES NOT EXIST
    # IT WILL BE CREATED WHEN SAVING
    # ======================================================

    if not store_output_folder.exists():

        print(
            f"📁 Store folder does not exist."
        )

        print(
            f"Creating folder: {store_output_folder}"
        )

        store_output_folder.mkdir(
            parents=True,
            exist_ok=True
        )


    try:

        # ==================================================
        # LOAD DATA
        # ==================================================

        print("Loading Data ...")

        df = load_processed_data(

            DATA_PATH,

            store_code,

            channel

        )

        print(
            f"✓ Data Loaded -> "
            f"{len(df):,} rows"
        )


        # ==================================================
        # PREPARE PROPHET
        # ==================================================

        prophet_df = prepare_prophet_data(
            df
        )

        (
            prophet_train,
            prophet_valid,
            prophet_test
        ) = train_validation_test_split(
            prophet_df
        )


        # ==================================================
        # PREPARE NHITS
        # ==================================================

        nhits_df = prepare_nhits_data(
            df
        )

        nhits_df = add_event_features(
            nhits_df
        )

        (
            nhits_train,
            nhits_valid,
            nhits_test
        ) = train_validation_test_split(
            nhits_df
        )


        # ==================================================
        # TRAIN PROPHET
        # ==================================================

        print("-" * 80)

        print(
            f"Training Prophet -> "
            f"{store_code}/{channel}"
        )

        print("-" * 80)


        prophet_model = train_prophet(
            prophet_train
        )


        print(
            "✓ Prophet Training Finished"
        )


        # ==================================================
        # PROPHET PREDICTION
        # ==================================================

        prophet_forecast = predict_prophet(

            prophet_model,

            prophet_valid

        )


        print(
            "✓ Prophet Prediction Finished"
        )


        # ==================================================
        # TRAIN NHITS
        # ==================================================

        print("-" * 80)

        print(
            f"Training NHITS -> "
            f"{store_code}/{channel}"
        )

        print("-" * 80)


        nhits_model = train_nhits(

            nhits_train,

            horizon=len(nhits_valid)

        )


        print(
            "✓ NHITS Training Finished"
        )


        # ==================================================
        # NHITS PREDICTION
        # ==================================================

        nhits_forecast = predict_nhits(

            nhits_model,

            nhits_valid

        )


        print(
            "✓ NHITS Prediction Finished"
        )


        # ==================================================
        # ENSEMBLE
        # ==================================================

        final_forecast = combine_forecasts(

            nhits_forecast,

            prophet_forecast,

            nhits_weight=0.2,

            prophet_weight=0.8

        )


        print(
            "✓ Ensemble Prediction Finished"
        )


        # ==================================================
        # METRICS
        # ==================================================

        prophet_metrics = evaluate_forecast(

            prophet_valid["y"],

            prophet_forecast["yhat"]

        )


        nhits_metrics = evaluate_forecast(

            nhits_valid["y"],

            nhits_forecast["yhat"]

        )


        ensemble_metrics = evaluate_forecast(

            prophet_valid["y"],

            final_forecast["yhat"]

        )


        # ==================================================
        # PRINT METRICS
        # ==================================================

        print("-" * 80)

        print(
            f"Prophet MAPE  : "
            f"{prophet_metrics['MAPE']:.2f}%"
        )

        print(
            f"NHITS MAPE    : "
            f"{nhits_metrics['MAPE']:.2f}%"
        )

        print(
            f"Ensemble MAPE : "
            f"{ensemble_metrics['MAPE']:.2f}%"
        )

        print("-" * 80)


        # ==================================================
        # SAVE RESULTS
        # ==================================================

        print(
            f"Saving Excel -> "
            f"{store_code}/{channel}.xlsx"
        )


        save_results_to_excel(

            output_path=output_file,

            store_name=store_code,

            channel=channel,

            valid_df=prophet_valid,

            prophet_forecast=prophet_forecast,

            nhits_forecast=nhits_forecast,

            final_forecast=final_forecast,

            prophet_metrics=prophet_metrics,

            nhits_metrics=nhits_metrics,

            ensemble_metrics=ensemble_metrics

        )


        print(
            f"✓ Excel Saved -> "
            f"{store_code}/{channel}.xlsx"
        )


        # ==================================================
        # VERIFY FILE WAS ACTUALLY CREATED
        # ==================================================

        if output_file.exists():

            print(
                f"✓ VERIFIED -> "
                f"{output_file}"
            )

            completed_jobs.append(
                (store_code, channel)
            )

        else:

            print(
                f"❌ Excel was not created!"
            )

            failed_jobs.append(
                (
                    store_code,
                    channel,
                    "Excel file was not created"
                )
            )


        # ==================================================
        # MEMORY CLEANUP
        # ==================================================

        del df

        del prophet_df
        del nhits_df

        del prophet_train
        del prophet_valid
        del prophet_test

        del nhits_train
        del nhits_valid
        del nhits_test

        del prophet_model
        del nhits_model

        del prophet_forecast
        del nhits_forecast

        del final_forecast

        gc.collect()


        print(
            f"✓ Finished -> "
            f"{store_code}/{channel}"
        )


    except Exception as e:

        print(
            f"❌ ERROR in "
            f"{store_code}/{channel}"
        )

        print(
            f"Error: {e}"
        )


        failed_jobs.append(

            (
                store_code,
                channel,
                str(e)
            )

        )


        # ==================================================
        # CLEAN MEMORY EVEN AFTER ERROR
        # ==================================================

        gc.collect()

        continue


# ==========================================================
# FINAL SUMMARY
# ==========================================================

print("\n")
print("=" * 100)
print("ALL STORES FINISHED")
print("=" * 100)


print(
    f"Total Jobs     : {total_jobs}"
)

print(
    f"Completed      : {len(completed_jobs)}"
)

print(
    f"Skipped        : {len(skipped_jobs)}"
)

print(
    f"Failed         : {len(failed_jobs)}"
)


# ==========================================================
# COMPLETED
# ==========================================================

if completed_jobs:

    print("\n")
    print("=" * 100)
    print("COMPLETED JOBS")
    print("=" * 100)

    for store_code, channel in completed_jobs:

        print(
            f"✓ {store_code}/{channel}"
        )


# ==========================================================
# SKIPPED
# ==========================================================

if skipped_jobs:

    print("\n")
    print("=" * 100)
    print("SKIPPED JOBS")
    print("=" * 100)

    for store_code, channel in skipped_jobs:

        print(
            f"⏭️ {store_code}/{channel}"
        )


# ==========================================================
# FAILED
# ==========================================================

if failed_jobs:

    print("\n")
    print("=" * 100)
    print("FAILED JOBS")
    print("=" * 100)

    for store_code, channel, error in failed_jobs:

        print(
            f"❌ {store_code}/{channel}"
        )

        print(
            f"   Error: {error}"
        )


print("\n")
print("=" * 100)
print("FINAL SUMMARY")
print("=" * 100)

print(
    f"✓ Completed : {len(completed_jobs)}"
)

print(
    f"⏭️ Skipped   : {len(skipped_jobs)}"
)

print(
    f"❌ Failed    : {len(failed_jobs)}"
)

print("=" * 100)

TOTAL STORES   : 35
TOTAL CHANNELS  : 6
TOTAL JOBS      : 210


[1/210]
Store   : ALX
Channel : store
⏭️ SKIPPED -> ALX/store
Already exists: /content/drive/MyDrive/Spinneys/Forecasting/FinalResults/ALX/store.xlsx


[2/210]
Store   : ALX
Channel : instashop
⏭️ SKIPPED -> ALX/instashop
Already exists: /content/drive/MyDrive/Spinneys/Forecasting/FinalResults/ALX/instashop.xlsx


[3/210]
Store   : ALX
Channel : talabat
Loading Data ...
✓ Data Loaded -> 1,270 rows
--------------------------------------------------------------------------------
Training Prophet -> ALX/talabat
--------------------------------------------------------------------------------


INFO:lightning_fabric.utilities.seed:Seed set to 1


✓ Prophet Training Finished
✓ Prophet Prediction Finished
--------------------------------------------------------------------------------
Training NHITS -> ALX/talabat
--------------------------------------------------------------------------------


INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.callbacks.model_summary:
  | Name                | Type          | Params | Mode 
--------------------------------------------------------------
0 | loss                | MAE           | 0      | train
1 | hist_cat_embeddings | ModuleList    | 0      | train
2 | futr_cat_embeddings | ModuleList    | 0      | train
3 | stat_cat_embeddings | ModuleList    | 0      | train
4 | padder_train        | ConstantPad1d | 0      | train
5 | scaler              | TemporalNorm  | 0      | train
6 | blocks              | ModuleList    | 1.1 M  | train
--------------------------------------------------------------
1.1 M     Trainable params
0         Non-trainable params
1.1 M     Total params
4.570     Total estimated model params size (MB)
46        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


... (213 outputs left out)

Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:
Detected KeyboardInterrupt, attempting graceful shutdown ...
ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/call.py", line 49, in _call_and_handle_interrupt
    return trainer_fn(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/trainer.py", line 598, in _fit_impl
    self._run(model, ckpt_path=ckpt_path)
  File "/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/trainer.py", line 1011, in _run
    results = self._run_stage()
              ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/trainer.py", line 1055, in _run_stage
    self.fit_loop.run()
  File "/usr/local/lib/python3.12/dist-packages/pytorch_lightning/loops/fit_loop.py", line 216, in run
    self.advance()
  File "/usr/local/lib/python3.12/dist-packages/pytorch_lightning/loops/fit_loop.py", line 458, in advance
    self.epoch_loop.run(self._data_fetcher)
  File "/usr/local/lib/python3.12

TypeError: object of type 'NoneType' has no len()

In [54]:
def save_results_to_excel(
    output_path,
    store_name,
    channel,
    valid_df,
    prophet_forecast,
    nhits_forecast,
    final_forecast,
    prophet_metrics,
    nhits_metrics,
    ensemble_metrics
):

    import pandas as pd
    import numpy as np
    import tempfile
    import os
    import matplotlib.pyplot as plt

    from openpyxl import load_workbook
    from openpyxl.styles import Font, PatternFill
    from openpyxl.utils import get_column_letter
    from openpyxl.drawing.image import Image

    # =====================================================
    # FORECAST TABLE
    # =====================================================

    forecast_df = pd.DataFrame({

        "Date": valid_df["ds"].values,

        "Actual": valid_df["y"].values,

        "Prophet Prediction":
            prophet_forecast["yhat"].values,

        "NHITS Prediction":
            (
                nhits_forecast["yhat"].values
                if nhits_forecast is not None
                else np.nan
            ),

        "Prediction":
            final_forecast["yhat"].values

    })

    # =====================================================
    # ERRORS
    # =====================================================

    forecast_df["Prophet Error"] = (
        forecast_df["Prophet Prediction"]
        -
        forecast_df["Actual"]
    )

    forecast_df["Prophet Percentage Error (%)"] = (

        forecast_df["Prophet Error"].abs()

        /

        forecast_df["Actual"].replace(0, np.nan)

    ) * 100


    # -----------------------------------------------------
    # NHITS ERROR
    # -----------------------------------------------------

    if nhits_forecast is not None:

        forecast_df["NHITS Error"] = (
            forecast_df["NHITS Prediction"]
            -
            forecast_df["Actual"]
        )

        forecast_df["NHITS Percentage Error (%)"] = (

            forecast_df["NHITS Error"].abs()

            /

            forecast_df["Actual"].replace(0, np.nan)

        ) * 100

    else:

        forecast_df["NHITS Error"] = np.nan

        forecast_df["NHITS Percentage Error (%)"] = np.nan


    # -----------------------------------------------------
    # ENSEMBLE ERROR
    # -----------------------------------------------------

    forecast_df["Error"] = (
        forecast_df["Prediction"]
        -
        forecast_df["Actual"]
    )

    forecast_df["Percentage Error (%)"] = (

        forecast_df["Error"].abs()

        /

        forecast_df["Actual"].replace(0, np.nan)

    ) * 100


    # =====================================================
    # TOTALS
    # =====================================================

    actual_sum = (
        forecast_df["Actual"].sum()
    )

    prophet_sum = (
        forecast_df["Prophet Prediction"].sum()
    )

    ensemble_sum = (
        forecast_df["Prediction"].sum()
    )


    if nhits_forecast is not None:

        nhits_sum = (
            forecast_df["NHITS Prediction"].sum()
        )

    else:

        nhits_sum = np.nan


    # =====================================================
    # SAFE DIFFERENCE FUNCTION
    # =====================================================

    def calculate_difference(forecast_total):

        if actual_sum == 0:

            return np.nan

        return forecast_total - actual_sum


    def calculate_difference_percentage(forecast_total):

        if actual_sum == 0:

            return np.nan

        return (
            (forecast_total - actual_sum)
            /
            actual_sum
        ) * 100


    # =====================================================
    # METRICS TABLE
    # =====================================================

    metrics_table = pd.DataFrame({

        "Metric": [

            "MAE",

            "RMSE",

            "MAPE (%)",

            "Average Error (%)",

            "Actual Total Sales",

            "Forecast Total Sales",

            "Difference",

            "Difference (%)",

            "Bias (%)"

        ],


        "Prophet": [

            prophet_metrics.get(
                "MAE",
                np.nan
            ),

            prophet_metrics.get(
                "RMSE",
                np.nan
            ),

            prophet_metrics.get(
                "MAPE",
                np.nan
            ),

            forecast_df[
                "Prophet Percentage Error (%)"
            ].mean(),

            actual_sum,

            prophet_sum,

            calculate_difference(
                prophet_sum
            ),

            calculate_difference_percentage(
                prophet_sum
            ),

            calculate_difference_percentage(
                prophet_sum
            )

        ],


        "NHITS": [

            (
                nhits_metrics.get(
                    "MAE",
                    np.nan
                )
                if nhits_forecast is not None
                else np.nan
            ),

            (
                nhits_metrics.get(
                    "RMSE",
                    np.nan
                )
                if nhits_forecast is not None
                else np.nan
            ),

            (
                nhits_metrics.get(
                    "MAPE",
                    np.nan
                )
                if nhits_forecast is not None
                else np.nan
            ),

            (
                forecast_df[
                    "NHITS Percentage Error (%)"
                ].mean()
                if nhits_forecast is not None
                else np.nan
            ),

            actual_sum,

            nhits_sum,

            (
                calculate_difference(
                    nhits_sum
                )
                if nhits_forecast is not None
                else np.nan
            ),

            (
                calculate_difference_percentage(
                    nhits_sum
                )
                if nhits_forecast is not None
                else np.nan
            ),

            (
                calculate_difference_percentage(
                    nhits_sum
                )
                if nhits_forecast is not None
                else np.nan
            )

        ],


        "Ensemble": [

            ensemble_metrics.get(
                "MAE",
                np.nan
            ),

            ensemble_metrics.get(
                "RMSE",
                np.nan
            ),

            ensemble_metrics.get(
                "MAPE",
                np.nan
            ),

            forecast_df[
                "Percentage Error (%)"
            ].mean(),

            actual_sum,

            ensemble_sum,

            calculate_difference(
                ensemble_sum
            ),

            calculate_difference_percentage(
                ensemble_sum
            ),

            calculate_difference_percentage(
                ensemble_sum
            )

        ]

    })


    # =====================================================
    # CREATE EXCEL
    # =====================================================

    with pd.ExcelWriter(

        output_path,

        engine="openpyxl"

    ) as writer:

        # -------------------------------------------------
        # Forecast
        # -------------------------------------------------

        forecast_df.to_excel(

            writer,

            sheet_name="Forecast",

            index=False

        )


        # -------------------------------------------------
        # Metrics
        # -------------------------------------------------

        metrics_table.to_excel(

            writer,

            sheet_name="Metrics",

            index=False

        )


    # =====================================================
    # LOAD WORKBOOK
    # =====================================================

    wb = load_workbook(

        output_path

    )


    # =====================================================
    # STYLING
    # =====================================================

    blue = PatternFill(

        start_color="4472C4",

        end_color="4472C4",

        fill_type="solid"

    )

    white = Font(

        bold=True,

        color="FFFFFF"

    )


    # =====================================================
    # STYLE ALL DATA SHEETS
    # =====================================================

    for sheet_name in [

        "Forecast",

        "Metrics"

    ]:

        ws = wb[sheet_name]

        # Header

        for cell in ws[1]:

            cell.fill = blue

            cell.font = white


        # Width

        for col in ws.columns:

            length = max(

                len(str(cell.value))

                if cell.value is not None

                else 0

                for cell in col

            )

            ws.column_dimensions[

                get_column_letter(
                    col[0].column
                )

            ].width = min(

                length + 4,

                35

            )


    # =====================================================
    # CREATE PLOT SHEET
    # =====================================================

    plot_ws = wb.create_sheet(

        "Plot"

    )


    # =====================================================
    # CREATE FIGURE
    # =====================================================

    fig, ax = plt.subplots(

        figsize=(18, 6)

    )


    # Actual

    ax.plot(

        valid_df["ds"],

        valid_df["y"],

        label="Actual",

        linewidth=3

    )


    # Prophet

    ax.plot(

        valid_df["ds"],

        prophet_forecast["yhat"],

        label="Prophet"

    )


    # NHITS ONLY IF AVAILABLE

    if nhits_forecast is not None:

        ax.plot(

            valid_df["ds"],

            nhits_forecast["yhat"],

            label="NHITS"

        )


    # Ensemble

    ax.plot(

        valid_df["ds"],

        final_forecast["yhat"],

        label="Ensemble"

    )


    ax.set_title(

        f"{store_name} - {channel} Forecast Comparison"

    )

    ax.legend()

    ax.grid(True)

    plt.tight_layout()


    # =====================================================
    # SAVE TEMPORARY IMAGE
    # =====================================================

    tmp_path = tempfile.mktemp(

        suffix=".png"

    )


    try:

        fig.savefig(

            tmp_path,

            dpi=200,

            bbox_inches="tight"

        )

        plt.close(fig)


        # -------------------------------------------------
        # ADD IMAGE
        # -------------------------------------------------

        img = Image(

            tmp_path

        )

        img.width = 1300

        img.height = 650

        plot_ws.add_image(

            img,

            "A1"

        )


        # -------------------------------------------------
        # SAVE WORKBOOK
        # -------------------------------------------------

        wb.save(

            output_path

        )


    finally:

        # -------------------------------------------------
        # DELETE TEMP IMAGE
        # -------------------------------------------------

        if os.path.exists(

            tmp_path

        ):

            os.remove(

                tmp_path

            )


    # =====================================================
    # FINAL MESSAGE
    # =====================================================

    print(

        f"✓ Saved -> "
        f"{store_name}/{channel}"
    )

In [56]:
from pathlib import Path
import gc


# ==========================================================
# PATHS
# ==========================================================

DATA_PATH = (
    "/content/drive/MyDrive/Spinneys/Forecasting/"
    "NewFinalResult"
)

OUTPUT_PATH = (
    "/content/drive/MyDrive/Spinneys/Forecasting/"
    "NewFinalResult"
)


# ==========================================================
# STORE TO RUN
# ==========================================================

STORE_TO_RUN = "AllStores"


# ==========================================================
# CHANNELS
# ==========================================================

channels = [

    "company_combined"

]


# ==========================================================
# CREATE STORE OUTPUT FOLDER
# ==========================================================

store_output_folder = (
    Path(OUTPUT_PATH) / STORE_TO_RUN
)

store_output_folder.mkdir(

    parents=True,

    exist_ok=True

)


# ==========================================================
# RUN CHANNELS
# ==========================================================

for channel in channels:

    print("\n")
    print("=" * 100)
    print(f"Store   : {STORE_TO_RUN}")
    print(f"Channel : {channel}")
    print("=" * 100)

    try:

        # ==================================================
        # OUTPUT FILE
        # ==================================================

        output_file = (

            store_output_folder

            /

            f"{channel}.xlsx"

        )


        # ==================================================
        # SKIP IF ALREADY EXISTS
        # ==================================================

        if output_file.exists():

            print(
                f"⏭️ SKIP {STORE_TO_RUN}/{channel}"
            )

            print(
                "Excel file already exists."
            )

            continue


        # ==================================================
        # CHECK INPUT FILE
        # ==================================================

        input_file = (

            Path(DATA_PATH)

            / STORE_TO_RUN

            / f"{channel}.parquet"

        )


        if not input_file.exists():

            print(
                f"⚠️ SKIP {STORE_TO_RUN}/{channel}"
            )

            print(
                f"Missing input file:"
            )

            print(input_file)

            continue


        # ==================================================
        # LOAD DATA
        # ==================================================

        print("Loading Data ...")

        df = load_processed_data(

            DATA_PATH,

            STORE_TO_RUN,

            channel

        )


        # ==================================================
        # PREPARE PROPHET
        # ==================================================

        prophet_df = prepare_prophet_data(

            df

        )


        prophet_train, prophet_valid, prophet_test = (

            train_validation_test_split(

                prophet_df

            )

        )


        # ==================================================
        # CHECK PROPHET DATA
        # ==================================================

        if prophet_train.empty:

            print(
                f"⚠️ SKIP {STORE_TO_RUN}/{channel}"
            )

            print(
                "Prophet training dataframe is empty."
            )

            continue


        if prophet_valid.empty:

            print(
                f"⚠️ SKIP {STORE_TO_RUN}/{channel}"
            )

            print(
                "Prophet validation dataframe is empty."
            )

            continue


        # ==================================================
        # PREPARE NHITS
        # ==================================================

        nhits_df = prepare_nhits_data(

            df

        )


        nhits_df = add_event_features(

            nhits_df

        )


        nhits_train, nhits_valid, nhits_test = (

            train_validation_test_split(

                nhits_df

            )

        )


        # ==================================================
        # TRAIN PROPHET
        # ==================================================

        print("-" * 80)

        print(
            f"Training Prophet -> "
            f"{STORE_TO_RUN}/{channel}"
        )

        print("-" * 80)


        prophet_model = train_prophet(

            prophet_train

        )


        print(
            "✓ Prophet Training Finished"
        )


        prophet_forecast = predict_prophet(

            prophet_model,

            prophet_valid

        )


        print(
            "✓ Prophet Prediction Finished"
        )


        # ==================================================
        # TRAIN NHITS
        # ==================================================

        NHITS_INPUT_SIZE = 365

        nhits_model = None

        nhits_forecast = None


        if len(nhits_train) < NHITS_INPUT_SIZE:

            print(
                f"⚠️ NHITS SKIPPED -> "
                f"{STORE_TO_RUN}/{channel}"
            )

            print(
                f"Available training timestamps: "
                f"{len(nhits_train)}"
            )

            print(
                f"Required training timestamps: "
                f"{NHITS_INPUT_SIZE}"
            )


        else:

            print("-" * 80)

            print(
                f"Training NHITS -> "
                f"{STORE_TO_RUN}/{channel}"
            )

            print("-" * 80)


            nhits_model = train_nhits(

                nhits_train,

                horizon=len(nhits_valid)

            )


            print(
                "✓ NHITS Training Finished"
            )


            nhits_forecast = predict_nhits(

                nhits_model,

                nhits_valid

            )


            print(
                "✓ NHITS Prediction Finished"
            )


        # ==================================================
        # FINAL FORECAST
        # ==================================================

        if nhits_forecast is None:

            print(
                "⚠️ NHITS unavailable -> "
                "Using Prophet as Ensemble"
            )

            final_forecast = (
                prophet_forecast.copy()
            )

        else:

            final_forecast = combine_forecasts(

                nhits_forecast,

                prophet_forecast,

                nhits_weight=0.2,

                prophet_weight=0.8

            )


        # ==================================================
        # METRICS
        # ==================================================

        prophet_metrics = evaluate_forecast(

            prophet_valid["y"],

            prophet_forecast["yhat"]

        )


        # --------------------------------------------------
        # NHITS METRICS
        # --------------------------------------------------

        if nhits_forecast is None:

            nhits_metrics = {

                "MAE": None,

                "RMSE": None,

                "MAPE": None

            }

        else:

            nhits_metrics = evaluate_forecast(

                nhits_valid["y"],

                nhits_forecast["yhat"]

            )


        # --------------------------------------------------
        # ENSEMBLE METRICS
        # --------------------------------------------------

        ensemble_metrics = evaluate_forecast(

            prophet_valid["y"],

            final_forecast["yhat"]

        )


        # ==================================================
        # SAVE EXCEL
        # ==================================================

        print("-" * 80)

        print(
            f"Saving Excel -> "
            f"{STORE_TO_RUN}/{channel}"
        )

        print("-" * 80)


        save_results_to_excel(

            output_path=output_file,

            store_name=STORE_TO_RUN,

            channel=channel,

            valid_df=prophet_valid,

            prophet_forecast=prophet_forecast,

            nhits_forecast=nhits_forecast,

            final_forecast=final_forecast,

            prophet_metrics=prophet_metrics,

            nhits_metrics=nhits_metrics,

            ensemble_metrics=ensemble_metrics

        )


        print(
            f"✓ Excel Saved -> "
            f"{output_file}"
        )


        # ==================================================
        # MEMORY CLEANUP
        # ==================================================

        del df

        del prophet_df

        del nhits_df

        del prophet_train

        del prophet_valid

        del prophet_test

        del nhits_train

        del nhits_valid

        del nhits_test

        del prophet_model

        del prophet_forecast

        del final_forecast

        nhits_model = None
        nhits_forecast = None

        gc.collect()


        print(
            f"✓ Finished "
            f"{STORE_TO_RUN}/{channel}"
        )


    except Exception as e:

        print(
            f"❌ ERROR in "
            f"{STORE_TO_RUN}/{channel}"
        )

        print(
            f"Error: {e}"
        )

        gc.collect()

        continue


# ==========================================================
# FINISHED
# ==========================================================

print("\n")

print("=" * 100)

print(
    f"STORE {STORE_TO_RUN} FINISHED"
)

print("=" * 100)



Store   : AllStores
Channel : company_combined
Loading Data ...
--------------------------------------------------------------------------------
Training Prophet -> AllStores/company_combined
--------------------------------------------------------------------------------


INFO:lightning_fabric.utilities.seed:Seed set to 1


✓ Prophet Training Finished
✓ Prophet Prediction Finished
--------------------------------------------------------------------------------
Training NHITS -> AllStores/company_combined
--------------------------------------------------------------------------------


INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.callbacks.model_summary:
  | Name                | Type          | Params | Mode 
--------------------------------------------------------------
0 | loss                | MAE           | 0      | train
1 | hist_cat_embeddings | ModuleList    | 0      | train
2 | futr_cat_embeddings | ModuleList    | 0      | train
3 | stat_cat_embeddings | ModuleList    | 0      | train
4 | padder_train        | ConstantPad1d | 0      | train
5 | scaler              | TemporalNorm  | 0      | train
6 | blocks              | ModuleList    | 1.1 M  | train
--------------------------------------------------------------
1.1 M     Trainable params
0         Non-trainable params
1.1 M     Total params
4.571     Total estimated model params size (MB)
46        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


... (7 outputs left out)

✓ NHITS Training Finished


/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting: |          | 0/? [00:00<?, ?it/s]

✓ NHITS Prediction Finished
--------------------------------------------------------------------------------
Saving Excel -> AllStores/company_combined
--------------------------------------------------------------------------------
✓ Saved -> AllStores/company_combined
✓ Excel Saved -> /content/drive/MyDrive/Spinneys/Forecasting/NewFinalResult/AllStores/company_combined.xlsx
✓ Finished AllStores/company_combined


STORE AllStores FINISHED


In [50]:
from pathlib import Path
import pandas as pd
import gc


def create_company_daily_dataset(
    input_path,
    output_path
):
    """
    Create company-level daily dataset by aggregating ONLY:

        store
        instashop
        talabat
        spinneys_delivery

    Excludes:
        combined
        loyal_store

    Output:
        One daily Parquet file containing company-wide metrics.
    """

    input_path = Path(input_path)
    output_path = Path(output_path)

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    # ==========================================================
    # CHANNELS TO INCLUDE
    # ==========================================================

    channels = [
        "store",
        "instashop",
        "talabat",
        "spinneys_delivery"
    ]

    # ==========================================================
    # COLUMNS
    # ==========================================================

    sum_columns = [
        "Total_Sales",
        "Loyal_Sales",
        "Non_Loyal_Sales",
        "Total_Transactions",
        "Loyal_Transactions",
        "Non_Loyal_Transactions",
        "Item_Quantity"
    ]

    # ==========================================================
    # STORE FOLDERS
    # ==========================================================

    stores = sorted([
        folder
        for folder in input_path.iterdir()
        if folder.is_dir()
    ])

    print("=" * 100)
    print("CREATING COMPANY DAILY DATASET")
    print("=" * 100)

    print(f"Input Path  : {input_path}")
    print(f"Output Path : {output_path}")
    print(f"Stores      : {len(stores)}")
    print()

    all_daily_data = []

    # ==========================================================
    # LOOP OVER STORES
    # ==========================================================

    for store_folder in stores:

        store_name = store_folder.name

        print("-" * 80)
        print(f"Processing Store: {store_name}")
        print("-" * 80)

        store_data = []

        # ======================================================
        # LOOP OVER FOUR CHANNELS ONLY
        # ======================================================

        for channel in channels:

            file_path = (
                store_folder /
                f"{channel}.parquet"
            )

            if not file_path.exists():

                print(
                    f"⚠️ Missing -> "
                    f"{store_name}/{channel}.parquet"
                )

                continue

            try:

                print(
                    f"  Loading -> {channel}.parquet"
                )

                df = pd.read_parquet(
                    file_path
                )

                # ------------------------------------------------
                # Keep only required columns
                # ------------------------------------------------

                required_columns = [
                    "transaction_date"
                ] + [
                    col
                    for col in sum_columns
                    if col in df.columns
                ]

                df = df[
                    required_columns
                ].copy()

                # ------------------------------------------------
                # Date
                # ------------------------------------------------

                df["transaction_date"] = pd.to_datetime(
                    df["transaction_date"],
                    errors="coerce"
                )

                df = df.dropna(
                    subset=["transaction_date"]
                )

                # ------------------------------------------------
                # Make sure numeric columns are numeric
                # ------------------------------------------------

                for col in sum_columns:

                    if col not in df.columns:

                        df[col] = 0

                    df[col] = pd.to_numeric(
                        df[col],
                        errors="coerce"
                    ).fillna(0)

                # ------------------------------------------------
                # Daily aggregation inside channel
                # ------------------------------------------------

                df = (
                    df
                    .groupby(
                        "transaction_date",
                        as_index=False
                    )[sum_columns]
                    .sum()
                )

                store_data.append(df)

                print(
                    f"  ✓ {channel}: "
                    f"{len(df):,} days"
                )

                del df
                gc.collect()

            except Exception as e:

                print(
                    f"  ❌ Error in "
                    f"{store_name}/{channel}"
                )

                print(
                    f"     {e}"
                )

        # ======================================================
        # IF NO CHANNELS AVAILABLE
        # ======================================================

        if not store_data:

            print(
                f"⚠️ No valid channels "
                f"for {store_name}"
            )

            continue

        # ======================================================
        # COMBINE FOUR CHANNELS FOR THIS STORE
        # ======================================================

        store_df = pd.concat(
            store_data,
            ignore_index=True
        )

        # ======================================================
        # AGGREGATE STORE
        # ======================================================

        store_df = (
            store_df
            .groupby(
                "transaction_date",
                as_index=False
            )[sum_columns]
            .sum()
        )

        # ======================================================
        # ADD STORE IDENTIFIER
        # ======================================================

        store_df["Store"] = store_name

        all_daily_data.append(
            store_df
        )

        print(
            f"✓ Finished Store: "
            f"{store_name}"
        )

        del store_data
        del store_df

        gc.collect()

    # ==========================================================
    # CHECK
    # ==========================================================

    if not all_daily_data:

        raise ValueError(
            "No valid data was found."
        )

    # ==========================================================
    # COMBINE ALL STORES
    # ==========================================================

    print()
    print("=" * 100)
    print("Combining All Stores")
    print("=" * 100)

    company_df = pd.concat(
        all_daily_data,
        ignore_index=True
    )

    # ==========================================================
    # COMPANY DAILY AGGREGATION
    # ==========================================================

    company_df = (
        company_df
        .groupby(
            "transaction_date",
            as_index=False
        )[sum_columns]
        .sum()
    )

    # ==========================================================
    # CREATE COMPLETE DAILY DATE RANGE
    # ==========================================================

    min_date = company_df[
        "transaction_date"
    ].min()

    max_date = company_df[
        "transaction_date"
    ].max()

    full_dates = pd.date_range(
        start=min_date,
        end=max_date,
        freq="D"
    )

    company_df = (
        company_df
        .set_index("transaction_date")
        .reindex(full_dates)
        .rename_axis("transaction_date")
        .reset_index()
    )

    # ==========================================================
    # FILL MISSING SALES / TRANSACTION VALUES WITH ZERO
    # ==========================================================

    for col in sum_columns:

        company_df[col] = (
            pd.to_numeric(
                company_df[col],
                errors="coerce"
            )
            .fillna(0)
        )

    # ==========================================================
    # CALCULATE DERIVED FEATURES
    # ==========================================================

    # ----------------------------------------------------------
    # Average Basket Size
    # ----------------------------------------------------------

    company_df["Avg_Basket_Size"] = 0.0

    mask_transactions = (
        company_df["Total_Transactions"] > 0
    )

    company_df.loc[
        mask_transactions,
        "Avg_Basket_Size"
    ] = (
        company_df.loc[
            mask_transactions,
            "Item_Quantity"
        ]
        /
        company_df.loc[
            mask_transactions,
            "Total_Transactions"
        ]
    )

    # ----------------------------------------------------------
    # Average Basket Price
    # ----------------------------------------------------------

    company_df["Avg_Basket_Price"] = 0.0

    mask_quantity = (
        company_df["Item_Quantity"] > 0
    )

    company_df.loc[
        mask_quantity,
        "Avg_Basket_Price"
    ] = (
        company_df.loc[
            mask_quantity,
            "Total_Sales"
        ]
        /
        company_df.loc[
            mask_quantity,
            "Item_Quantity"
        ]
    )

    # ==========================================================
    # TIME FEATURES
    # ==========================================================

    company_df["Day_of_Week"] = (
        company_df[
            "transaction_date"
        ].dt.dayofweek + 1
    )

    company_df["Week_of_Year"] = (
        company_df[
            "transaction_date"
        ].dt.isocalendar()
        .week
        .astype(int)
    )

    company_df["Month"] = (
        company_df[
            "transaction_date"
        ].dt.month
    )

    company_df["Is_Weekend"] = (
        company_df[
            "Day_of_Week"
        ]
        .isin([6, 7])
        .astype(int)
    )

    # ==========================================================
    # SORT
    # ==========================================================

    company_df = company_df.sort_values(
        "transaction_date"
    ).reset_index(
        drop=True
    )

    # ==========================================================
    # FINAL COLUMN ORDER
    # ==========================================================

    final_columns = [

        "transaction_date",

        "Total_Sales",

        "Loyal_Sales",

        "Non_Loyal_Sales",

        "Total_Transactions",

        "Loyal_Transactions",

        "Non_Loyal_Transactions",

        "Item_Quantity",

        "Avg_Basket_Size",

        "Avg_Basket_Price",

        "Day_of_Week",

        "Week_of_Year",

        "Month",

        "Is_Weekend"

    ]

    company_df = company_df[
        final_columns
    ]

    # ==========================================================
    # SAVE PARQUET
    # ==========================================================

    print()
    print("=" * 100)
    print("Saving Company Dataset")
    print("=" * 100)

    company_df.to_parquet(
        output_path,
        index=False
    )

    # ==========================================================
    # SUMMARY
    # ==========================================================

    print()
    print("=" * 100)
    print("✓ COMPANY DATASET CREATED")
    print("=" * 100)

    print(
        f"Date Range : "
        f"{company_df['transaction_date'].min().date()} "
        f"-> "
        f"{company_df['transaction_date'].max().date()}"
    )

    print(
        f"Days       : "
        f"{len(company_df):,}"
    )

    print(
        f"Total Sales: "
        f"{company_df['Total_Sales'].sum():,.2f}"
    )

    print(
        f"Output     : "
        f"{output_path}"
    )

    print()
    print("Columns:")

    print(
        company_df.columns.tolist()
    )

    return company_df

In [52]:
INPUT_PATH = (
    "/content/drive/MyDrive/Spinneys/Forecasting/"
    "total_Processed_Data"
)

OUTPUT_PATH = (
    "/content/drive/MyDrive/Spinneys/Forecasting/"
    "NewFinalResult/AllStores/company_combined.parquet"
)

company_df = create_company_daily_dataset(
    input_path=INPUT_PATH,
    output_path=OUTPUT_PATH
)

CREATING COMPANY DAILY DATASET
Input Path  : /content/drive/MyDrive/Spinneys/Forecasting/total_Processed_Data
Output Path : /content/drive/MyDrive/Spinneys/Forecasting/NewFinalResult/AllStores/company_combined.parquet
Stores      : 35

--------------------------------------------------------------------------------
Processing Store: ALX
--------------------------------------------------------------------------------
  Loading -> store.parquet
  ✓ store: 1,275 days
  Loading -> instashop.parquet
  ✓ instashop: 1,275 days
  Loading -> talabat.parquet
  ✓ talabat: 1,270 days
  Loading -> spinneys_delivery.parquet
  ✓ spinneys_delivery: 1,275 days
✓ Finished Store: ALX
--------------------------------------------------------------------------------
Processing Store: ALX2
--------------------------------------------------------------------------------
  Loading -> store.parquet
  ✓ store: 792 days
  Loading -> instashop.parquet
  ✓ instashop: 560 days
  Loading -> talabat.parquet
  ✓ talaba